# 11 — Error Analysis of the Severity Engine & VADER

**Input:** `data/labels/label_queue.csv` — 200 manually-labeled reviews (blind ground truth) merged with model predictions.

**Goal:** Turn the validation gaps into a concrete fix list for `src/analysis.py`. We quantify:
1. **Critical misses** — human severity 4–5 but model predicted 1–3.
2. **Negation-related sentiment errors** — VADER flips on negated phrases.
3. **Financial-crisis keywords the model missed** — terms present in missed reviews that never trigger high severity.
4. **Top keywords in misclassified reviews** — vocabulary the lexicon should cover.


In [1]:
import sys, os, re
from collections import Counter
import pandas as pd, numpy as np

sys.path.append(os.path.abspath(".."))
df = pd.read_csv("../data/labels/label_queue.csv")
df["true_severity"] = df["true_severity"].astype(int)
df["severity_score"] = df["severity_score"].astype(int)
print(f"Labeled reviews: {len(df)}")
df[["review_text","vader_sentiment","true_sentiment","severity_score","true_severity"]].head()

Labeled reviews: 200


,review_text,vader_sentiment,true_sentiment,severity_score,true_severity
0,"No fees, so many features. Chime Rocks",Negative,Positive,1,1
1,hater,Negative,Negative,1,5
2,Very useful app as long as I was sending money...,Positive,Negative,5,4
3,I hate the fact that you have to send minimum ...,Negative,Negative,5,5
4,not so good,Negative,Negative,1,4


## 0. Headline agreement metrics

In [2]:
sent_agree = (df["vader_sentiment"].str.lower() == df["true_sentiment"].str.lower()).mean()
sev_exact  = (df["severity_score"] == df["true_severity"]).mean()
sev_within1 = (abs(df["severity_score"] - df["true_severity"]) <= 1).mean()
print(f"Sentiment agreement : {sent_agree:.1%}")
print(f"Severity exact      : {sev_exact:.1%}")
print(f"Severity within +/-1: {sev_within1:.1%}")

Sentiment agreement : 67.0%
Severity exact      : 20.0%
Severity within +/-1: 48.0%


## 1. Critical misses (human 4–5, model 1–3)

In [3]:
crit = df[(df["true_severity"]>=4) & (df["severity_score"]<=3)].copy()
n_high = (df["true_severity"]>=4).sum()
print(f"Critical misses: {len(crit)} of {n_high} high-severity reviews ({len(crit)/n_high:.1%})")
print("Predicted severity among misses:", crit["severity_score"].value_counts().sort_index().to_dict())
print(f"-> {(crit['severity_score']==1).sum()} defaulted to severity=1 (NO keyword matched at all)")
crit[["review_text","true_severity","severity_score"]].head(10)

Critical misses: 44 of 87 high-severity reviews (50.6%)
Predicted severity among misses: {1: 33, 2: 1, 3: 10}
-> 33 defaulted to severity=1 (NO keyword matched at all)


,review_text,true_severity,severity_score
1,hater,5,1
4,not so good,4,1
5,"garbage,I feel like an app that's accepted wor...",5,1
6,Custom Service sucks 😕 A man from India made m...,4,1
7,sucks,4,1
8,Usually didn't require help but when needed he...,4,1
10,Automatically installed itself onto my phone a...,5,1
12,"Out of all the online banks out there, ""chime""...",5,2
13,Chime did not explain anything to me at all an...,5,1
14,"sometime transactions go smoothly, sometimes t...",4,1


## 2. Negation-related sentiment errors

In [4]:
neg_words = ["not","no","never","cant","can't","cannot","won't","wont","doesn't","doesnt",
             "didn't","didnt","isn't","isnt","wasn't","wasnt","n't"]
def has_negation(t):
    t = str(t).lower()
    return any(w in t for w in neg_words)

sent_wrong = df[df["vader_sentiment"].str.lower() != df["true_sentiment"].str.lower()].copy()
sent_wrong["neg"] = sent_wrong["review_text"].apply(has_negation)
missed_neg = sent_wrong[(sent_wrong["true_sentiment"]=="Negative") & (sent_wrong["vader_sentiment"]!="Negative")]
print(f"Sentiment errors: {len(sent_wrong)}")
print(f"  containing negation: {sent_wrong['neg'].sum()} ({sent_wrong['neg'].mean():.0%})")
print(f"  missed negatives (human Neg, VADER not Neg): {len(missed_neg)}")
missed_neg[["review_text","vader_sentiment","true_sentiment"]].head(10)

Sentiment errors: 66
  containing negation: 48 (73%)
  missed negatives (human Neg, VADER not Neg): 46


,review_text,vader_sentiment,true_sentiment
2,Very useful app as long as I was sending money...,Positive,Negative
5,"garbage,I feel like an app that's accepted wor...",Positive,Negative
8,Usually didn't require help but when needed he...,Positive,Negative
13,Chime did not explain anything to me at all an...,Neutral,Negative
14,"sometime transactions go smoothly, sometimes t...",Positive,Negative
15,"Adding money to your account, they take it out...",Positive,Negative
17,This app was not working. It refused to displa...,Positive,Negative
18,why when I change my virtual number old charge...,Positive,Negative
20,Too much fees!,Neutral,Negative
29,PayPal is Good application but after Linking m...,Positive,Negative


## 3. Financial-crisis keywords the model missed

In [5]:
crisis_terms = ["money","funds","account","locked","lock","stolen","steal","fraud","scam","access",
 "frozen","freeze","hacked","hack","unauthorized","dispute","charged","charge","refund","deposit",
 "withdraw","transfer","pending","hold","held","closed","close","block","suspend","verify","verification",
 "cant","can't","cannot","won't","unable","missing","gone","lost","disappeared","waiting","days","weeks"]
blob = " ".join(crit["review_text"].astype(str).str.lower())
counts = Counter({t: len(re.findall(r"\b"+re.escape(t)+r"\b", blob)) for t in crisis_terms})
counts = {t:c for t,c in counts.items() if c>0}
pd.Series(counts).sort_values(ascending=False).head(20)

money       20
account     19
can't       10
hold         4
unable       3
hacked       3
days         3
dispute      2
deposit      2
steal        2
close        2
transfer     2
funds        1
access       1
cant         1
verify       1
won't        1
lost         1
dtype: int64

## 4. Top keywords in severity-misclassified reviews (off by ≥2)

In [6]:
sev_wrong = df[abs(df["true_severity"]-df["severity_score"])>=2].copy()
STOP = set("the a an and or but to of for in on is it my me i you we was were be been this that with at as have has had not no so they your get from would are like out now one even when".split())
words = re.findall(r"[a-z']+", " ".join(sev_wrong["review_text"].astype(str).str.lower()))
words = [w for w in words if w not in STOP and len(w)>2]
print(f"{len(sev_wrong)} reviews misclassified by >=2 severity levels")
pd.Series(Counter(words)).sort_values(ascending=False).head(25)

104 reviews misclassified by >=2 severity levels


app        45
money      43
account    35
chime      23
can't      16
use        16
card       16
cash       15
i'm        14
paypal     12
venmo      11
service    11
because    10
can        10
good       10
help       10
will       10
all         9
easy        9
new         9
make        9
don't       9
it's        9
using       9
over        9
dtype: int64

## 5. Fix list for `src/analysis.py`

Concrete, evidence-backed changes:

1. **Add emphatic/profanity negatives** — `hater`, `garbage`, `sucks`, `shady`, `trash`, `worst`, `useless`, `terrible`, `horrible`, `ripoff` (many one-word rants defaulted to severity=1).
2. **Handle negation for severity** — phrases like `not working`, `not so good`, `wouldn't trust`, `can't access`, `unable to` should escalate. Add negated-verb patterns rather than only positive keywords.
3. **Expand the financial-crisis lexicon** — `hold`/`held`, `hacked`, `steal`/`stolen`, `dispute`, `deposit`, `days`/`weeks` (waiting), `disappeared`, `won't let me`. These appear repeatedly in critical misses.
4. **Fix the default** — 33 of 44 critical misses defaulted to severity=1 because *no* keyword matched. Consider (a) a fallback that couples VADER's strong-negative compound (≤ −0.5) with low ratings to bump severity, and (b) lowering the default when rating ≤ 2.
5. **Negation-aware VADER** — 73% of sentiment errors contain negation; 46 true-negatives were missed. Consider preprocessing or a rating-aware backstop for the `is_hidden_negative` flag.
